In [ ]:
import itertools
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline

from src.data import get_electrode_df

In [ ]:
window_size = 0.1
stride = 0.05
outdir = "."

scores_path = "/userdata/jgauthier/projects/ideal-word-representations/outputs/encoders/timit-no_repeats/"
speech_responsive_threshold = 0.025

roi_filters = {
    "stg": ["superiortemporal"],
    "mtg": ["middletemporal"],
    "precentral": ["precentral"],
    "postcentral": ["postcentral"],
    "smg": ["supramarginal"],
}

In [ ]:
all_epoch_paths = list(Path("epochs").glob("*.fif"))

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epochs", str(path))[0]
    if subject_name == "EC282":
        # missing ecog data
        continue
    epochs[subject_name] = mne.read_epochs(str(path)).pick("ecog").resample(100)

In [ ]:
electrode_df = pd.concat([get_electrode_df(subject_name) for subject_name in epochs.keys()],
                         keys=epochs.keys(), names=["subject"])

In [ ]:
scores_paths = {subject_name: Path(scores_path) / "baseline" / subject_name / "scores.csv" for subject_name in epochs.keys()}
scores_paths = {subject_name: path for subject_name, path in scores_paths.items() if path.exists()}
scores_df = pd.concat([pd.read_csv(path) for path in scores_paths.values()],
                      keys=scores_paths.keys(), names=["subject"]).droplevel(-1) \
    .rename(columns={"output_dim": "electrode_idx"}).set_index("electrode_idx", append=True)
scores_df = pd.merge(scores_df, electrode_df.roi.astype(str), left_index=True, right_index=True)

In [ ]:
speech_responsive_electrodes = scores_df.groupby(["subject", "roi", "electrode_idx"]).score.mean()
speech_responsive_electrodes = speech_responsive_electrodes[speech_responsive_electrodes > speech_responsive_threshold].sort_values(ascending=False)
speech_responsive_electrodes

In [ ]:
def run_decoding_analysis(epochs, stride, window_size, roi_filter=None):
    global_tmin = 0. # min([epoch.times.min() for epoch in epochs.values()])
    global_tmax = max([epoch.times.max() for epoch in epochs.values()])
    windows_left = np.arange(global_tmin, global_tmax, stride)
    windows_right = np.minimum(global_tmax, windows_left + window_size)
    windows = list(zip(windows_left, windows_right))

    scores = {}
    phoneme_pairs = next(iter(epochs.values())).metadata.phoneme_pair.unique()

    for phoneme_pair, (tmin, tmax), subject_name in tqdm(list(itertools.product(phoneme_pairs, windows, epochs))):
        try:
            speech_responsive_electrodes_i = speech_responsive_electrodes.loc[subject_name]
        except KeyError:
            continue

        if roi_filter is not None:
            try:
                speech_responsive_electrodes_i = speech_responsive_electrodes_i.loc[roi_filter]
            except KeyError:
                continue
        speech_responsive_electrodes_i = speech_responsive_electrodes_i.index.get_level_values("electrode_idx")
        if len(speech_responsive_electrodes_i) == 0:
            continue

        epochs_i = epochs[subject_name][f"phoneme_pair == '{phoneme_pair}'"].copy()
        pick_electrodes = list(set(speech_responsive_electrodes_i) & set(range(len(epochs_i.ch_names))))

        epochs_i = epochs_i.pick(pick_electrodes).crop(tmin, tmax)
        if len(epochs_i) == 0:
            continue

        # epochs * channels * time
        X = epochs_i.get_data()
        X = X.reshape(X.shape[0], -1)

        y = epochs_i.metadata.word_end.str[0] == epochs_i.metadata.phoneme_pair.str[0]

        cv_inner = StratifiedKFold(3, shuffle=True)
        cv_outer = StratifiedKFold(3, shuffle=True)

        model = make_pipeline(StandardScaler(), PCA(n_components=0.95),
                            LogisticRegressionCV(Cs=10, cv=cv_inner, max_iter=1000))
        scores_i = cross_val_score(model, X, y, cv=cv_outer, scoring="roc_auc")

        scores[subject_name, phoneme_pair, tmin, tmax] = scores_i
        print(subject_name, phoneme_pair, tmin, tmax, scores_i.mean())

    return scores

In [ ]:
decoding_kwargs = dict(stride=stride, window_size=window_size)
all_scores = {"*": run_decoding_analysis(epochs, **decoding_kwargs)}

In [ ]:
for name, rois in tqdm(roi_filters.items(), unit="ROI"):
    print(name)
    all_scores[name] = run_decoding_analysis(epochs, roi_filter=rois, **decoding_kwargs)

In [ ]:
scores_df = pd.concat(
    {roi_filter: pd.concat(
        {key: pd.Series(scores_i).rename("roc_auc") for key, scores_i in scores.items()},
        names=["subject", "phoneme_pair", "tmin", "tmax", "fold"])
     for roi_filter, scores in all_scores.items() if len(scores) > 0},
    names=["roi_filter"])

In [ ]:
scores_df.to_csv(Path(outdir) / "scores.csv")

In [ ]:
plot_df = scores_df.reset_index()
plot_df["t_center"] = plot_df.tmin + (plot_df.tmax - plot_df.tmin) / 2

In [ ]:
g = sns.relplot(data=plot_df, x="t_center", y="roc_auc",
                hue="subject", hue_order=sorted(epochs.keys()),
                row="roi_filter", col="phoneme_pair",
                kind="line", errorbar="se", aspect=2)
for ax in g.axes.flat:
    ax.axhline(0.5, ls="--", color="gray")
    ax.set_ylim(0.4, 1.0)
    ax.set_ylabel("ROC AUC")
    ax.set_xlabel("Time from word onset (s)")

In [ ]:
g.savefig(Path(outdir) / "decoding.pdf")